# Independent group draw and percentile reproduction


Expand every saved whole-group draw to original paired gene rows. Count the six numerators/denominators independently, then reproduce linear percentiles with scalar sorted interpolation. No calls to the estimation implementation; exact comparisons only. Intervals are conditional on recorded groups and frozen source predictions, not biological independence or calibrated per-gene probabilities.

In [1]:
import hashlib,json,math,numpy as np
from pathlib import Path
def sha(path):
 with Path(path).open('rb') as stream:return hashlib.file_digest(stream,'sha256').hexdigest()
receipts=0
for line in (PACKET/'SHA256SUMS.txt').read_text().splitlines():
 digest,name=line.split('  ',1);path=PACKET/name
 assert not path.is_symlink() and path.resolve().is_relative_to(PACKET.resolve())
 assert sha(path)==digest,name
 receipts+=1
inputs=json.loads((PACKET/'input_manifest.json').read_text())
for path,digest in inputs.items():assert sha(path)==digest,path
report=json.loads((ROOT/'starplast/data/functional_agreement.json').read_text())
state=json.loads((PACKET/'uncertainty.json').read_text())
arrays=np.load(PACKET/'draws.npz')
assert state['entities']==152 and state['recorded_groups']==145
assert sum(not row['training_supported'] for row in report['rows'])==4
assert state['group_order']==sorted({row['group'] for row in report['rows']})
by_group={group:[row for row in report['rows'] if row['group']==group] for group in state['group_order']}
manual=[]
for i,draw in enumerate(arrays['selections']):
 rows=[row for g in draw for row in by_group[state['group_order'][int(g)]]]
 calls=[(r['left_prediction'],r['right_prediction'],r['truth']) for r in rows]
 joint=sum(x is not None and y is not None for x,y,t in calls)
 agree=sum(x is not None and x==y for x,y,t in calls)
 recovered=sum(x==y==t for x,y,t in calls)
 wrong=sum(x is not None and y is not None and x!=t and y!=t for x,y,t in calls)
 delta=sum(y==t for x,y,t in calls)-sum(x==t for x,y,t in calls)
 numerators=[joint,agree,recovered,recovered,wrong,delta]
 denominators=[len(rows),joint,agree,len(rows),joint,len(rows)]
 assert arrays['numerators'][i].tolist()==numerators
 assert arrays['denominators'][i].tolist()==denominators
 values=[n/d if d else float('nan') for n,d in zip(numerators,denominators)]
 np.testing.assert_array_equal(arrays['ratios'][i],values)
 manual.append(values)
names=['joint_coverage','agreement_among_joint_calls','source_recovery_among_agreements','correct_agreements_all_eligible','shared_wrong_among_joint_calls','right_minus_left_recovery_all_eligible']
for i,name in enumerate(names):
 values=sorted(row[i] for row in manual if math.isfinite(row[i]))
 bounds=[]
 for q in (.025,.975):
  rank=(len(values)-1)*q;lo=math.floor(rank);hi=math.ceil(rank);weight=rank-lo
  difference=values[hi]-values[lo]
  bounds.append(values[hi]-difference*(1-weight) if weight>=.5 else values[lo]+difference*weight)
 assert bounds==state['rates'][name]['interval'],name
 assert len(values)==state['rates'][name]['valid_draws']
 assert 500-len(values)==state['rates'][name]['undefined_draws']
export=json.loads((PACKET/'scorecard.json').read_text())['snapshot']
assert export['counts']==report['counts'] and export['rates']==report['rates']
assert export['group_uncertainty']==state
assert export['calibrated_confidence'] is None and export['biological_admission'] is False
result={'outputs_verified':receipts,'current_inputs_verified':len(inputs),'all500draw_counts_ratios_and_six_intervals_reproduced_exactly':True,'genes':152,'unsupported':4,'recorded_groups':145,'draws':500,'agreement_recovery_interval':state['rates']['source_recovery_among_agreements']['interval'],'shared_wrong_interval':state['rates']['shared_wrong_among_joint_calls']['interval'],'right_minus_left_recovery_interval':state['rates']['right_minus_left_recovery_all_eligible']['interval'],'fits_performed':0,'biological_admission':False}
(OUT/'acceptance.json').write_text(json.dumps(result,indent=2)+'\n')
result

{'outputs_verified': 10, 'current_inputs_verified': 6, 'all500draw_counts_ratios_and_six_intervals_reproduced_exactly': True, 'genes': 152, 'unsupported': 4, 'recorded_groups': 145, 'draws': 500, 'agreement_recovery_interval': [0.25, 0.45809487951807226], 'shared_wrong_interval': [0.4609573879358052, 0.6299544072948328], 'right_minus_left_recovery_interval': [0.05263157894736842, 0.20465645695364235], 'fits_performed': 0, 'biological_admission': False}